In [ ]:
TAK -3 : Mall E-commerce: Data Quality Assessment & Exploratory Data Analysis (EDA)
Dataset: mall_ecommerce_messy_dataset.csv
Tools: Python, Pandas, NumPy, Jupyter Notebook

This notebook assesses data quality, investigates duplicate records, compares performance across categories, and identifies the highest-priced product. It ends with business interpretations and recommended actions.

Important: The dataset is intentionally messy. Inspect records before removing or changing them.

In [ ]:
Division 1 — Load and understand the dataset
What: Load the CSV and inspect its dimensions, sample records, columns, and data types.
Why: We need to understand the dataset before drawing conclusions.
Outcome: A baseline view of the data and its potential quality issues.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("mall_ecommerce_messy_dataset.csv")

print("Rows and columns:", df.shape)
display(df.head(10))
df.info()
display(df.describe(include="all").T)

Rows and columns: (123, 12)


,Order_ID,Order_Date,Mall,City,Product,Category,Quantity,Unit_Price_INR,Discount,Sales_Channel,Customer_Age,Customer_Rating
0,ORD-0001,2025-05-10,Mall Central,Chennai,Coffee Mug,Home & Living,1.0,244.43,0.15,Click & Collect,62.0,4.2
1,ORD-0002,2025-08-09,mall central,Bengaluru,Water Bottle,Home & Living,2.0,455.93,0.00,In-Store,57.0,4.5
2,ORD-0003,15/03/2025,Mall Central,Mumbai,Keyboard,Electronics,4.0,1772.22,0.00,In-Store,37.0,4.8
3,ORD-0004,2025-06-13,Riverside Mall,Chennai,Power Bank,Electronics,6.0,1424.78,0.15,In-Store,41.0,3.1
4,ORD-0005,NaN,City Square,Bengaluru,USB-C Cable,Electronics,1.0,259.87,0.10,Click & Collect,61.0,4.1
5,ORD-0006,2025-01-21,Mall Central,Mumbai,Power Bank,Electronics,2.0,1242.31,0.15,In-Store,68.0,4.7
6,ORD-0007,2025-04-20,Grand Arcade,Pune,Keyboard,Electronics,-3.0,1634.20,0.15,In-Store,42.0,2.6
7,ORD-0008,2025-10-04,NaN,Hyderabad,Desk Lamp,Home & Living,4.0,1264.84,0.00,In-Store,56.0,4.9
8,ORD-0009,2025-01-23,CITY SQUARE,Pune,T-Shirt,Fashion,3.0,671.84,0.05,Online,42.0,3.0
9,ORD-0010,2025-04-10,City Square,Bengaluru,Water Bottle,Home & Living,4.0,396.09,0.00,Online,29.0,4.2


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123 entries, 0 to 122
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order_ID         123 non-null    object 
 1   Order_Date       120 non-null    object 
 2   Mall             120 non-null    object 
 3   City             121 non-null    object 
 4   Product          121 non-null    object 
 5   Category         123 non-null    object 
 6   Quantity         121 non-null    float64
 7   Unit_Price_INR   121 non-null    float64
 8   Discount         121 non-null    float64
 9   Sales_Channel    121 non-null    object 
 10  Customer_Age     120 non-null    float64
 11  Customer_Rating  120 non-null    float64
dtypes: float64(5), object(7)
memory usage: 11.7+ KB


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Order_ID,123,120,ORD-0031,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Order_Date,120,101,2025-01-02,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Mall,120,10,Grand Arcade,28,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,121,8,Chennai,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Product,121,17,Keyboard,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Category,123,6,Electronics,45,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,121.0,NaN,NaN,NaN,14.157025,93.03288,-3.0,2.0,4.0,6.0,999.0
Unit_Price_INR,121.0,NaN,NaN,NaN,1973.866364,9023.645532,-250.0,455.93,1084.43,1717.84,99999.0
Discount,121.0,NaN,NaN,NaN,0.108264,0.166752,0.0,0.0,0.1,0.15,1.75
Sales_Channel,121,5,In-Store,46,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
Division 2 — Identify and analyze missing values
What: Count missing values and calculate their percentage for each column.
Why: Missing values can bias summaries, reduce usable records, and affect business decisions.
Outcome: A ranked missing-data report and suitable recommendations for each affected field.

In [2]:
# Count missing values in each column
missing_count = df.isnull().sum()

# Calculate percentage of missing values in each column
missing_percent = (df.isna().sum() / len(df) * 100).round(2)

missing_report = pd.DataFrame({
    "Missing_Count": missing_count,
    "Missing_Percent": missing_percent
}).sort_values("Missing_Percent", ascending=False)

display(missing_report)

,Missing_Count,Missing_Percent
Order_Date,3,2.44
Mall,3,2.44
Customer_Age,3,2.44
Customer_Rating,3,2.44
City,2,1.63
Product,2,1.63
Quantity,2,1.63
Unit_Price_INR,2,1.63
Discount,2,1.63
Sales_Channel,2,1.63


In [3]:
# Show columns that have at least one missing value
columns_to_clean = missing_report[missing_report["Missing_Count"] > 0]
display(columns_to_clean)

,Missing_Count,Missing_Percent
Order_Date,3,2.44
Mall,3,2.44
Customer_Age,3,2.44
Customer_Rating,3,2.44
City,2,1.63
Product,2,1.63
Quantity,2,1.63
Unit_Price_INR,2,1.63
Discount,2,1.63
Sales_Channel,2,1.63


In [ ]:
Business interpretation: Missing order dates can prevent time-based sales reporting; missing prices or quantities affect revenue; missing product/category labels weaken product comparisons. Before filling values, investigate why they are missing. Use dropna() only when the affected records are unusable and removal is justified; use fillna() when a defensible value can be estimated or assigned.

In [4]:
# Demonstrate alternatives without overwriting the original df.
# Option A: drop rows missing a critical field (example only).
df_with_date = df.dropna(subset=["Order_Date"])
print("Rows retained after dropping records with missing Order_Date:", len(df_with_date), "of", len(df))

# Option B: fill a numeric field with its median (example copy).
df_fill_demo = df.copy()
df_fill_demo["Unit_Price_INR"] = pd.to_numeric(df_fill_demo["Unit_Price_INR"], errors="coerce")
median_price = df_fill_demo["Unit_Price_INR"].median()
df_fill_demo["Unit_Price_INR"] = df_fill_demo["Unit_Price_INR"].fillna(median_price)
print("Median used for missing prices in demo:", median_price)

# Choose the strategy based on context; don't apply every option automatically.

Rows retained after dropping records with missing Order_Date: 120 of 123
Median used for missing prices in demo: 1084.43


In [ ]:
Division 3 — Detect duplicate records
What: Identify exact duplicate rows, repeated order IDs, and the number of times each ID appears.
Why: Duplicate transactions can inflate sales, but repeated IDs can also represent conflicting versions or valid line items.
Outcome: A review list that separates exact duplicates from records requiring investigation.

In [5]:
# Exact duplicate rows
exact_duplicate_mask = df.duplicated()
print("Number of exact duplicate rows:", exact_duplicate_mask.sum())
display(df.loc[exact_duplicate_mask])

# Repeated Order_ID values
repeated_order_mask = df["Order_ID"].duplicated(keep=False)
repeated_orders = df.loc[repeated_order_mask].sort_values("Order_ID")

print("Rows with repeated Order_ID:", len(repeated_orders))
display(repeated_orders)

# Count occurrences of each Order_ID
order_id_counts = df["Order_ID"].value_counts()
display(order_id_counts[order_id_counts > 1])

Number of exact duplicate rows: 1


,Order_ID,Order_Date,Mall,City,Product,Category,Quantity,Unit_Price_INR,Discount,Sales_Channel,Customer_Age,Customer_Rating
120,ORD-0012,2025-09-10,Metro Plaza,Mumbai,Backpack,Accessories,6.0,1636.05,0.2,Online,38.0,3.2


Rows with repeated Order_ID: 6


,Order_ID,Order_Date,Mall,City,Product,Category,Quantity,Unit_Price_INR,Discount,Sales_Channel,Customer_Age,Customer_Rating
11,ORD-0012,2025-09-10,Metro Plaza,Mumbai,Backpack,Accessories,6.0,1636.05,0.20,Online,38.0,3.2
120,ORD-0012,2025-09-10,Metro Plaza,Mumbai,Backpack,Accessories,6.0,1636.05,0.20,Online,38.0,3.2
20,ORD-0021,2025-05-06,Mall Central,Mumbai,Sunglasses,Fashion,7.0,970.94,0.15,Click & Collect,60.0,2.9
121,ORD-0021,2025-05-06,MALL CENTRAL,Mumbai,Sunglasses,Fashion,7.0,970.94,0.15,Click & Collect,60.0,2.9
30,ORD-0031,2025-07-08,Grand Arcade,Pune,Water Bottle,Home & Living,1.0,416.94,0.10,Click & Collect,22.0,3.5
122,ORD-0031,2025-07-08,Grand Arcade,Pune,Water Bottle,Home & Living,5.0,416.94,0.10,Click & Collect,22.0,3.5


Order_ID
ORD-0031    2
ORD-0012    2
ORD-0021    2
Name: count, dtype: int64

In [ ]:
Exact duplicate rows may have entered through repeated imports or resubmitted transactions. Repeated order IDs with different quantities or text formatting should be reviewed against source records. Remove exact duplicates only when they are confirmed accidental; do not blindly delete every repeated ID.

In [6]:
# Safe demonstration: create a separate version with exact duplicate rows removed.
df_no_exact_duplicates = df.drop_duplicates().copy()
print("Rows before:", len(df))
print("Rows after removing exact duplicate rows:", len(df_no_exact_duplicates))

# Repeated IDs remain visible for review.
display(
    df_no_exact_duplicates[
        df_no_exact_duplicates["Order_ID"].duplicated(keep=False)
    ].sort_values("Order_ID")
)

Rows before: 123
Rows after removing exact duplicate rows: 122


,Order_ID,Order_Date,Mall,City,Product,Category,Quantity,Unit_Price_INR,Discount,Sales_Channel,Customer_Age,Customer_Rating
20,ORD-0021,2025-05-06,Mall Central,Mumbai,Sunglasses,Fashion,7.0,970.94,0.15,Click & Collect,60.0,2.9
121,ORD-0021,2025-05-06,MALL CENTRAL,Mumbai,Sunglasses,Fashion,7.0,970.94,0.15,Click & Collect,60.0,2.9
30,ORD-0031,2025-07-08,Grand Arcade,Pune,Water Bottle,Home & Living,1.0,416.94,0.10,Click & Collect,22.0,3.5
122,ORD-0031,2025-07-08,Grand Arcade,Pune,Water Bottle,Home & Living,5.0,416.94,0.10,Click & Collect,22.0,3.5


In [ ]:
Division 4 — Group-wise analysis by product category
What: Group orders by Category and calculate order counts, units sold, average price, and sales revenue.
Why: Category-level comparisons help identify high-performing and underperforming areas.
Outcome: A ranked performance table to guide inventory, promotion, and merchandising decisions.

In [7]:
# Use a separate analysis copy and convert numeric fields safely.
eda = df_no_exact_duplicates.copy()

for col in ["Quantity", "Unit_Price_INR", "Discount"]:
    eda[col] = pd.to_numeric(eda[col], errors="coerce")

# Standardize category labels for analysis.
eda["Category"] = eda["Category"].astype("string").str.strip().str.title()

# Revenue is calculated only where the necessary fields are present.
eda["Revenue"] = (
    eda["Quantity"] * eda["Unit_Price_INR"] * (1 - eda["Discount"])
)

category_summary = eda.groupby("Category", dropna=False).agg(
    Order_Rows=("Order_ID", "count"),
    Units_Sold=("Quantity", "sum"),
    Average_Unit_Price=("Unit_Price_INR", "mean"),
    Median_Unit_Price=("Unit_Price_INR", "median"),
    Total_Revenue=("Revenue", "sum")
).sort_values("Total_Revenue", ascending=False)

display(category_summary.round(2))

,Order_Rows,Units_Sold,Average_Unit_Price,Median_Unit_Price,Total_Revenue
Category,,,,,
Electronics,46,410.0,3665.24,1731.14,710684.39
Fashion,27,107.0,1430.23,1000.90,139639.67
Accessories,18,72.0,1124.15,1440.59,77497.41
Stationery,7,1025.0,77.55,78.31,76505.94
Home & Living,24,93.0,689.05,428.32,60260.00
